In [ ]:
# Analysis workflow
# This notebook combines analyses for the EJ water study:
# 1. Exposure footprint: summarize the number and population of watersheds exposed to toxconc by period.
# 2. Demographic disparity: compare demographic composition for impacted versus non-impacted watersheds using Welch t-tests.
# 3. Spatial agreement: assess overlap in impacted watersheds across the three study periods.
# 4. Disparity ratios: compute weighted exposure statistics for each region and study period.
# The workflow loads HUC12 spatial boundaries, joins them with previoulsy processed RSEI and demographic tables,
# and writes the resulting summaries to CSV outputs.

ERROR 1: PROJ: proj_create_from_database: Open of /users/2/oboiko/.conda/envs/geo/share/proj failed


In [ ]:
import geopandas as gpd
import pandas as pd
from scipy import stats
from utils.config import root_dir
from utils.disparity_ratio import compute_weighted_exposure_stats

# Data preparation
regions = [
    "Headwaters",
    "Gorge",
    "Driftless",
    "Working River",
    "Confluence",
    "Chickasaw",
    "Delta",
    "Lower Mississippi",
    "Gulf South",
    "Total",
]
periods = ["2008_2012", "2013_2017", "2018_2022"]


def load_and_prepare_data():
    gdf = gpd.read_file(f"{root_dir}/results/aoi_huc12_boundaries.gpkg")
    huc12_rsei = pd.read_csv(
        f"{root_dir}/results/huc12_rsei_toxconc_weighted.csv",
        dtype={"huc12": str},
        index_col=0,
    )
    huc12_demographics = pd.read_csv(
        f"{root_dir}/results/huc12_demographics.csv",
        dtype={"huc12": str},
        index_col=0,
    )
    rsei_cols = [
        col for col in huc12_rsei.columns if col != "huc12" and col not in gdf.columns
    ]
    demo_cols = [
        col
        for col in huc12_demographics.columns
        if col != "huc12" and col not in gdf.columns
    ]
    gdf = gdf.merge(huc12_rsei[["huc12"] + rsei_cols], on="huc12", how="left")
    gdf = gdf.merge(huc12_demographics[["huc12"] + demo_cols], on="huc12", how="left")
    return gdf


gdf = load_and_prepare_data()

### Footprint of exposure

In [ ]:
def summarize_exposure_footprint(gdf, regions, periods):
    rows = []
    for region in regions:
        subset = (
            gdf.copy() if region == "Total" else gdf[gdf["Region"] == region].copy()
        )
        for period in periods:
            tox_col = f"{period}_TOXCONC"
            pop_col = f"{period}_total"
            subset_notnull = subset.dropna(subset=[tox_col])
            rows.append(
                {
                    "Region": region,
                    "Period": period,
                    "n_total": len(subset),
                    "n_exposed": len(subset_notnull),
                    "pop_total": subset[pop_col].sum(),
                    "pop_exposed": subset_notnull[pop_col].sum(),
                }
            )

    df_stats = pd.DataFrame(rows)
    df_stats["Region"] = pd.Categorical(
        df_stats["Region"], categories=regions, ordered=True
    )
    return (
        df_stats.pivot(index="Region", columns="Period")
        .swaplevel(0, 1, axis=1)
        .sort_index(axis=1)
    )


df_pivot = summarize_exposure_footprint(gdf, regions, periods)
df_pivot.to_csv(f"{root_dir}results/toxconc_descriptive_stats.csv")

### T-Test on impacted vs non-impacted watersheds

In [ ]:
base_variables = [
    "share_nonhsp_white",
    "share_black",
    "share_native",
    "share_asian",
    "share_hispanic",
    "share_below_poverty",
    "share_2_above_poverty",
]


def summarize_t_tests(gdf, periods, base_variables):
    results = []
    for period in periods:
        var_cols = [f"{period}_{v}" for v in base_variables]
        tox_col = f"{period}_TOXCONC"
        is_impacted = gdf[tox_col].notnull()

        df_imp = gdf.loc[is_impacted, var_cols]
        df_nonimp = gdf.loc[~is_impacted, var_cols]

        avg_imp = df_imp.mean() * 100
        avg_nonimp = df_nonimp.mean() * 100
        t_stats, p_vals = stats.ttest_ind(
            df_imp, df_nonimp, equal_var=False, nan_policy="omit"
        )

        for col, t_stat, p_val in zip(var_cols, t_stats, p_vals):
            results.append(
                {
                    "variable": col,
                    "avg_impacted_pct": avg_imp[col],
                    "avg_nonimpacted_pct": avg_nonimp[col],
                    "t_stat": t_stat,
                    "p_val": p_val,
                }
            )

    return pd.DataFrame(results).set_index("variable")


ttest_summary = summarize_t_tests(gdf, periods, base_variables)
ttest_summary.to_csv(f"{root_dir}results/toxconc_ttest_stats.csv")

### Spatial agreement between impacted watersheds across time

In [ ]:
def summarize_spatial_agreement(gdf, cols):
    impact_mask = gdf[cols].notnull()
    intersection = impact_mask.all(axis=1).sum()
    union = impact_mask.any(axis=1).sum()
    jaccard_score = intersection / union if union > 0 else 0
    return {
        "jaccard_score": jaccard_score,
        "agreement_area": intersection,
        "total_impacted_footprint": union,
    }


agreement_summary = summarize_spatial_agreement(
    gdf,
    ["2008_2012_TOXCONC", "2013_2017_TOXCONC", "2018_2022_TOXCONC"],
)
print(agreement_summary)

{'jaccard_score': np.float64(0.8185185185185185),
 'agreement_area': np.int64(663),
 'total_impacted_footprint': np.int64(810)}

### Disparity Ratios

In [ ]:
def compute_disparity_ratios_for_periods(base_gdf, periods):
    results = []
    for study_period in periods:
        metric = f"{study_period}_TOXCONC"
        gdf_period = base_gdf.copy()
        for region in regions[:-1]:
            subset = gdf_period[gdf_period["Region"] == region]
            population_examined = subset[f"{study_period}_total"].sum()
            analysis = f"Regional: {region}, N = {len(subset)}, population = {population_examined}"
            result = compute_weighted_exposure_stats(
                subset,
                study_period=study_period,
                metric=metric,
            )
            result["analysis"] = analysis
            result["study_period"] = study_period
            results.append(result)

    return pd.concat(results, ignore_index=True)


disparity_results = compute_disparity_ratios_for_periods(gdf, periods)
for study_period in periods:
    outfilepth = f"{root_dir}results/disparity_ratios_{study_period}.csv"
    subset = disparity_results[disparity_results["study_period"] == study_period]
    subset.to_csv(outfilepth, index=False)
    print(f"Saved results for {study_period} to {outfilepth}")

Saved results for 2008_2012 to /projects/standard/lenkne/oboiko/ejwater//results/disparity_ratios_2008_2012.csv
Saved results for 2013_2017 to /projects/standard/lenkne/oboiko/ejwater//results/disparity_ratios_2013_2017.csv
Saved results for 2018_2022 to /projects/standard/lenkne/oboiko/ejwater//results/disparity_ratios_2018_2022.csv
